# Import Primary Libraries

In [74]:
import pandas as pd
import numpy as np
import joblib
import sys
from pathlib import Path

# Import functions from src/features.py
sys.path.append(
    str(Path().resolve().parent)
)
from src.features import (
    team_rating
)

# Loading Data and Model

In [75]:
fixtures = pd.read_csv("../data/processed/wc2026_matches.csv")
elo_ratings = pd.read_csv("../data/processed/elo_clean.csv")
players = pd.read_csv("../data/processed/players_clean.csv")

model = joblib.load("best_model.pkl")

In [76]:
# Renaming country column to host_country for clarity
fixtures.rename(columns={"country": "host_country"}, inplace=True)

# Features

## Squad Ratings
The squad ratings are derived from FC26. To prevent data leakage, they were not used as features in model training. Thus, they will not directly be used as features during simulation as well. 

In [77]:
squad_requirements = { # General squad requirements
    "GK": 3,
    "DEF": 9,
    "MID": 7,
    "FWD": 7
}
squad_ratings = team_rating(players, requirements=squad_requirements, rating_name="squad_rating")

top11_requirements = { # 4-3-3 formation as estimate for starting 11 requirements
    "GK": 1,
    "DEF": 4,
    "MID": 3,
    "FWD": 3
}
top11_ratings = team_rating(players, requirements=top11_requirements, rating_name="top11_rating")

In [78]:
# Rename columns before merger
squad_ratings.rename(columns={"avg_rating": "squad_rating"}, inplace=True)
top11_ratings.rename(columns={"avg_rating": "top11_rating"}, inplace=True)

# Merge ratings with fixtures
fixtures = fixtures.merge(
    squad_ratings[["country", "squad_rating"]], 
    left_on="home_team", 
    right_on="country", 
    how="left").rename(columns={"squad_rating": "home_squad_rating"}).drop(columns=["country"])

fixtures = fixtures.merge(
    squad_ratings[["country", "squad_rating"]], 
    left_on="away_team", 
    right_on="country", 
    how="left").rename(columns={"squad_rating": "away_squad_rating"}).drop(columns=["country"])

fixtures = fixtures.merge(
    top11_ratings[["country", "top11_rating"]], 
    left_on="home_team", 
    right_on="country", 
    how="left").rename(columns={"top11_rating": "home_top11_rating"}).drop(columns=["country"])

fixtures = fixtures.merge(
    top11_ratings[["country", "top11_rating"]], 
    left_on="away_team", 
    right_on="country", 
    how="left").rename(columns={"top11_rating": "away_top11_rating"}).drop(columns=["country"])

fixtures

,date,home_team,away_team,home_score,away_score,tournament,city,host_country,neutral,home_squad_rating,away_squad_rating,home_top11_rating,away_top11_rating
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,75.576923,59.384615,77.181818,66.818182
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,73.576923,75.576923,76.090909,77.090909
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,72.884615,72.807692,77.000000,76.272727
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,76.423077,73.692308,79.090909,75.181818
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,68.307692,77.615385,72.090909,80.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,52.346154,82.115385,55.545455,84.454545
68,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,77.346154,81.961538,79.454545,84.636364
69,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,71.384615,53.730769,75.727273,58.818182
70,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,59.500000,83.461538,68.363636,85.727273


## Elo features

In [79]:
# Checking for any discrepancies in country names between fixtures and elo ratings
fixtures_countries = set(fixtures["home_team"]).union(
    set(fixtures["away_team"])
)

elo_countries = set(elo_ratings["country_full"])

print("In fixtures but not elo:")
print(sorted(fixtures_countries - elo_countries))

print("\nIn elo but not fixtures:")
print(sorted(elo_countries - fixtures_countries))

In fixtures but not elo:
[]

In elo but not fixtures:
['Afghanistan', 'Albania', 'American Samoa', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Armenia', 'Aruba', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia', 'Botswana', 'British Virgin Islands', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Cayman Islands', 'Central African Republic', 'Chad', 'Chile', 'China PR', 'Chinese Taipei', 'Comoros', 'Congo', 'Cook Islands', 'Costa Rica', 'Cuba', 'Cyprus', 'Czechoslovakia', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Faroe Islands', 'Fiji', 'Finland', 'Gabon', 'Georgia', 'Gibraltar', 'Greece', 'Grenada', 'Guam', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Honduras', 'Hong Kong, China', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Israel', 'Italy', 'Jamaica', 'Kazakh

In [ ]:
elo_ratings_wc2026 = elo_ratings[elo_ratings["country_full"].isin(fixtures_countries)]
elo_ratings_wc2026 = elo_ratings_wc2026[elo_ratings_wc2026["rank_date"] == "2026-04-01"] # last update to elo ratings before the World Cup 2026
print(len(elo_ratings_wc2026))

48
